# Phase 4D — Public GitHub Release Live Demo

This notebook downloads the four verified assets from the public **v1.0-final** GitHub Release. **Google Drive is not required.** Run all cells, wait for the MiniLM model to load, then use the interactive Top-10 search widget. First-time setup needs internet access; the live query itself uses the saved local index.


In [ ]:
%pip -q install "numpy>=2.0,<2.3" "scikit-learn>=1.6,<2" "sentence-transformers>=3,<6" "joblib>=1.4,<2" "ipywidgets>=8,<9" "threadpoolctl>=3.5,<4"
from pathlib import Path
import json,time,html,os,hashlib,urllib.request,tempfile
import numpy as np
import joblib
from sklearn.preprocessing import normalize
from threadpoolctl import threadpool_limits
from sentence_transformers import SentenceTransformer
from IPython.display import display,HTML,clear_output
import ipywidgets as widgets

# Public GitHub Release: no Drive mount or private access needed.
ROOT=Path('/content/HP-SS-Engine')
TAG='v1.0-final'
BASE=f'https://github.com/OsmanTurgut2335/HP-SS-Engine/releases/download/{TAG}/'
ASSETS={
 'qwen_corpus_20k_progress.jsonl':('data/raw/qwen_corpus_20k_progress.jsonl',10830847,'00fa40516cb2934e797dfb52e1906ccfcd395d489cf6f5213bc272b2a8940a25'),
 'qwen_20k_minilm_384_float32.npy':('phase2_matrix_ops/2A_sparse_dense/qwen_20k_minilm_384_float32.npy',30720128,'207e8271de8d8787a1ac4694ca9369721e38ad47d0190a573d08fa0f2510dea9'),
 'truncated_svd_k100_full_20k.joblib':('phase4_final_system/models/truncated_svd_k100_full_20k.joblib',155543,'d220f7378fa80c95fa8527dc4439c28061707fbd052dd2e8afcc300e89337e68'),
 'reduced_index_k100_full_20k.npy':('phase4_final_system/models/reduced_index_k100_full_20k.npy',8000128,'d269d94082c84dfd0838b61a8faeeac83f51fb01b8bbec8959f5352aba5b8321')
}
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()
for name,(relative,size,digest) in ASSETS.items():
    target=ROOT/relative
    if not (target.is_file() and target.stat().st_size==size and sha256(target)==digest):
        target.parent.mkdir(parents=True,exist_ok=True)
        print('Downloading:',name,flush=True)
        with urllib.request.urlopen(urllib.request.Request(BASE+name,headers={'User-Agent':'HP-SS-Engine/1.0'}),timeout=120) as src,open(str(target)+'.part','wb') as dst:
            while chunk:=src.read(1024*1024):
                dst.write(chunk)
        os.replace(str(target)+'.part',target)
    assert target.stat().st_size==size and sha256(target)==digest,'Release asset integrity check failed: '+name
    print('Verified:',name)
CORPUS=ROOT/'data/raw/qwen_corpus_20k_progress.jsonl'
MODEL=ROOT/'phase4_final_system/models/truncated_svd_k100_full_20k.joblib'
INDEX=ROOT/'phase4_final_system/models/reduced_index_k100_full_20k.npy'
print('All four public GitHub Release assets downloaded and SHA-256 verified; no Drive required.')


## 1. Load the prepared index and real source documents

This step can take time, so do it **before** the presentation. It is not part of the live 30 seconds.

In [ ]:
with open(CORPUS,'r',encoding='utf-8') as f:
    docs=[json.loads(line) for line in f if line.strip()]
assert len(docs)==20000,f'Expected 20K documents; got {len(docs)}'
svd=joblib.load(MODEL)
index=np.ascontiguousarray(np.load(INDEX),dtype=np.float32)
assert index.shape==(20000,100)
norms=np.linalg.norm(index,axis=1)
assert np.allclose(norms,1.0,atol=1e-4),'Expected pre-normalized SVD index; check Phase 4A.'
encoder=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device='cpu')
print(f'READY: {len(docs):,} real documents | {index.shape[1]}d SVD index | MiniLM CPU encoder')
print('First document fields:',list(docs[0].keys()))

## 2. One real search, with end-to-end timing

A query is encoded with the **same MiniLM model** used for the corpus. We transform its 384d embedding with the **saved SVD model**, normalize it, compute cosine similarities against all 20K rows, then return the highest-scoring ten documents. We do **not** include data loading or model startup in per-query latency.

The display shows a short excerpt of each actual Qwen response, plus document ID and cosine score. We show *separate* encoding and search times so the instructor can distinguish user-visible response time from benchmarked vector-search time.

In [ ]:
def run_search(query,top_k=10):
    if not query or not query.strip():raise ValueError('Enter a nonempty English query.')
    start=time.perf_counter()
    vector=encoder.encode([query.strip()],convert_to_numpy=True,normalize_embeddings=False,show_progress_bar=False)
    after_encoding=time.perf_counter()
    reduced=svd.transform(np.asarray(vector,dtype=np.float32))
    reduced=normalize(reduced,axis=1).astype(np.float32,copy=False)
    after_projection=time.perf_counter()
    with threadpool_limits(limits=1):
        scores=index@reduced[0]
        best=np.lexsort((np.arange(len(scores)),-scores))[:top_k]
    after_search=time.perf_counter()
    results=[]
    for i in best:
        d=docs[int(i)]
        results.append({'row':int(i),'id':str(d.get('id',i)),
                        'score':float(scores[i]),
                        'prompt':str(d.get('prompt','')),
                        'response':str(d.get('response',d.get('text','')))})
    after_results=time.perf_counter()
    return {'query':query,'results':results,
            'encoding_ms':(after_encoding-start)*1000,
            'svd_ms':(after_projection-after_encoding)*1000,
            'search_ms':(after_search-after_projection)*1000,
            'result_prep_ms':(after_results-after_search)*1000,
            'total_ms':(after_results-start)*1000}

warmup=run_search('How can machine learning help detect cybersecurity threats?',top_k=3)
print('Warmup passed (not a benchmark):',round(warmup['total_ms'],2),'ms')
print('Ready for interactive demonstration.')


## 3. LIVE DEMO — only this widget is needed on screen

**30-second presentation:** Enter a new English query, press Search, show actual top-10 results and real measured latency. The query is not hardcoded and can be chosen by the instructor.

**Optional rehearsal:** Run the example once, then replace it with another query during the presentation. If Colab disconnects, reconnect and run setup cells again; screenshots/videos are only a backup, not a substitute for the required live demo.

In [ ]:
query_box=widgets.Text(value='How can AI help detect cyber attacks?',
    placeholder='Type a question in English',description='Query:',
    layout=widgets.Layout(width='95%'))
search_button=widgets.Button(description='Search Top 10',button_style='success',
    icon='search',layout=widgets.Layout(width='180px'))
status=widgets.HTML(value='<span style="color:#0f766e">● Ready — 20,000 documents loaded</span>')
results_box=widgets.Output(layout=widgets.Layout(width='95%'))

def metric_card(label,value):
    return ('<div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:10px;'
            'padding:12px;min-width:135px;flex:1">'
            f'<div style="font-size:12px;color:#475569">{html.escape(label)}</div>'
            f'<div style="font-size:21px;font-weight:700;color:#0f172a">{value:.2f} ms</div></div>')

def render_results(result):
    metrics=[('Total query',result['total_ms']),('MiniLM encoding',result['encoding_ms']),
             ('SVD transform',result['svd_ms']),('Search + Top-10',result['search_ms']),
             ('Result preparation',result['result_prep_ms'])]
    cards=''.join(metric_card(label,value) for label,value in metrics)
    items=[]
    for rank,r in enumerate(result['results'],1):
        excerpt=' '.join(r['response'].split()) or '[Empty response]'
        if len(excerpt)>260:excerpt=excerpt[:257]+'...'
        items.append(
            '<div style="border:1px solid #e2e8f0;border-radius:10px;'
            'padding:12px 14px;margin-top:9px;background:white">'
            f'<div style="font-weight:700;color:#0f172a">#{rank} &nbsp; Document {html.escape(r["id"])}'
            f'<span style="float:right;color:#0f766e">Cosine {r["score"]:.4f}</span></div>'
            f'<div style="color:#334155;line-height:1.5;margin-top:7px">{html.escape(excerpt)}</div></div>')
    display(HTML(
        '<div style="font-family:Arial,sans-serif;max-width:1000px">'
        f'<h3 style="margin:8px 0">Query: {html.escape(result["query"])}</h3>'
        '<div style="display:flex;flex-wrap:wrap;gap:8px;margin:12px 0">'
        +cards+'</div><div style="font-size:12px;color:#64748b;margin:8px 0">'
        'Total includes embedding, SVD, vector search, ranking and result preparation. '
        'Model loading and HTML rendering are excluded. '
        'Top-10 is approximate relative to original 384d cosine.</div>'
        '<h4>Top 10 matching documents</h4>'+''.join(items)+'</div>'))

def handle_search(_):
    search_button.disabled=True
    status.value='<span>Searching…</span>'
    with results_box:
        clear_output(wait=True)
        try:
            result=run_search(query_box.value,10)
            render_results(result)
            status.value='<span style="color:#0f766e">● Search complete — real results</span>'
        except Exception as exc:
            print(type(exc).__name__+':',exc)
            status.value='<span style="color:#b91c1c">● Search failed — inspect output</span>'
        finally:
            search_button.disabled=False

search_button.on_click(handle_search)
display(widgets.VBox([
    widgets.HTML('<h3>HP-SS Engine | Live Semantic Search</h3>'
                 '<p>20,000 real Qwen documents · MiniLM 384d → SVD 100d · Top-10 cosine</p>'),
    query_box,widgets.HBox([search_button,status]),results_box
]))
print('Enter a new English query and click Search Top 10.')


## Instructor presentation notes (not part of the 30 seconds)

- **Problem:** find top-10 semantically similar texts from 20K real Qwen responses.
- **Baseline:** Phase 1 pure Python cosine; be explicit that its TF-IDF workload differs from MiniLM and use Phase 3D same-vector comparison for fair speedup claims.
- **Journey:** vectorization and BLAS, batching, SVD dimensionality reduction, parallelism comparisons.
- **Result:** Phase 4C full batch median at 20K was 190.79 ms for 100 queries using SVD batched 100d; **the single live query has its own measured time**, displayed above. Phase 4B external-query SVD Top-10 overlap was 77%.
- **Lessons:** discuss measured behavior and variability; don't claim a specific bottleneck without evidence.
- **10× data:** 200K×384 float32 ~293 MiB for raw embeddings alone; extrapolation is not a measured latency.
- **Live demo:** run setup in advance; during the demo only interact with the final search widget. Aim for ~30 seconds of presentation time.
- **Reliability:** if results look irrelevant, investigate semantic model and dataset characteristics; do not handpick fake answers.

After pilot verification, archive notebook and final screenshot/results as evidence, then build the final presentation story and report.